# 01. 서울 전월세 시장 데이터 파이프라인 (2023–2025)

**목적:** 서울 25개 자치구의 월별 아파트 전월세 **계약 건수(시장 활동)**를 국토교통부 실거래, KOSIS 인구이동, 한국은행 기준금리, 서울 생활인구와 결합하여 머신러닝용 `ml_rent_market` 테이블을 생성합니다.

> 이 프로젝트의 target은 실제 **빈방 수/공실률**이 아닙니다. 관측 가능한 임대차 계약 건수를 이용해 다음 달 임대시장 활동을 예측합니다.

실행 순서: **환경설정 → MOLIT 수집/캐시 → 정제/MySQL → KOSIS → ECOS → 생활인구 → 4개 데이터 JOIN → 검증**


## 0. 사전 준비

노트북은 프로젝트의 `notebooks/` 폴더에서 실행하는 것을 기준으로 합니다.

필요 파일:
- `../.env` — `MOLIT_API_KEY`, `DB_HOST`, `DB_PORT`, `DB_USER`, `DB_PASSWORD`, `DB_NAME`
- `../data/raw/kosis_migration_2023_2025.csv`
- `../data/raw/ecos_base_rate_2023_2025.csv`
- `../data/raw/LOCAL_PEOPLE_GU_2023.csv`
- `../data/raw/LOCAL_PEOPLE_GU_2024.csv`
- `../data/raw/LOCAL_PEOPLE_GU_2025.csv`

### CSV 주의사항
- KOSIS: UTF-8, **2단 헤더**이므로 `header=[0, 1]` 사용
- ECOS: UTF-8, 월 컬럼이 `2023/01` 형태인지 확인
- 서울 생활인구 2023: **CP949 + 한글 핵심 컬럼명**
- 서울 생활인구 2024: UTF-8 + 영문 컬럼명
- 서울 생활인구 2025: CP949 + 영문 컬럼명
- Excel로 CSV를 다시 저장하면 인코딩/헤더/숫자 형식이 변할 수 있으므로 원본 파일을 보존합니다.


In [10]:
import os
import time
from pathlib import Path
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env", override=True)

API_KEY = os.getenv("MOLIT_API_KEY")
required_db = ["DB_HOST", "DB_PORT", "DB_USER", "DB_PASSWORD", "DB_NAME"]
missing = [k for k in required_db if not os.getenv(k)]
if not API_KEY:
    raise ValueError(".env에 MOLIT_API_KEY가 없습니다.")
if missing:
    raise ValueError(f".env에 DB 설정이 없습니다: {missing}")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("환경변수 및 MySQL 연결 확인 완료")


환경변수 및 MySQL 연결 확인 완료


## 1. 국토교통부 아파트 전월세 실거래 데이터

서울 25개 자치구 × 2023–2025년을 수집합니다. 이미 3개년 정제 CSV가 있으면 API를 다시 호출하지 않고 캐시를 사용합니다. 공공데이터 API 키는 URL 인코딩된 키이므로 이 프로젝트에서는 `serviceKey`를 URL에 직접 연결합니다.

API 자료에는 거래 고유 ID가 없으므로 **완전히 같은 행이 있더라도 임의로 `drop_duplicates()` 하지 않습니다.** 서로 다른 실제 계약일 수 있기 때문입니다.


In [11]:
seoul_gu = {
    "종로구":"11110", "중구":"11140", "용산구":"11170", "성동구":"11200", "광진구":"11215",
    "동대문구":"11230", "중랑구":"11260", "성북구":"11290", "강북구":"11305", "도봉구":"11320",
    "노원구":"11350", "은평구":"11380", "서대문구":"11410", "마포구":"11440", "양천구":"11470",
    "강서구":"11500", "구로구":"11530", "금천구":"11545", "영등포구":"11560", "동작구":"11590",
    "관악구":"11620", "서초구":"11650", "강남구":"11680", "송파구":"11710", "강동구":"11740"
}
YEARS = [2023, 2024, 2025]
BASE_URL = "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"
RAW_RENT = RAW / "seoul_rent_2023_2025.csv"
CLEAN_RENT = PROCESSED / "seoul_rent_2023_2025_clean.csv"


In [12]:
def fetch_rent_month(gu_name, gu_code, ymd, num_rows=5000):
    rows = []
    page = 1
    while True:
        url = (
            f"{BASE_URL}?serviceKey={API_KEY}"
            f"&LAWD_CD={gu_code}&DEAL_YMD={ymd}"
            f"&numOfRows={num_rows}&pageNo={page}"
        )
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        root = ET.fromstring(response.text)
        result_code = root.findtext(".//resultCode")
        if result_code not in (None, "000", "00"):
            raise RuntimeError(f"API 오류 {gu_name} {ymd}: {result_code} {root.findtext('.//resultMsg')}")
        items = root.findall(".//item")
        for item in items:
            row = {child.tag: child.text for child in item}
            row["guName"] = gu_name
            rows.append(row)
        total = int(root.findtext(".//totalCount") or len(rows))
        if len(rows) >= total or not items:
            break
        page += 1
        time.sleep(0.2)
    return rows


def clean_rent(df):
    out = df.copy()
    out["deposit"] = out["deposit"].astype(str).str.replace(",", "", regex=False)
    out["deposit"] = pd.to_numeric(out["deposit"], errors="coerce")
    out["monthlyRent"] = pd.to_numeric(out["monthlyRent"], errors="coerce")
    for col in ["excluUseAr", "floor", "buildYear", "dealYear", "dealMonth", "dealDay"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["dealDate"] = pd.to_datetime(
        dict(year=out["dealYear"], month=out["dealMonth"], day=out["dealDay"]), errors="coerce"
    )
    out["rentType"] = out["monthlyRent"].apply(lambda x: "전세" if x == 0 else "월세")
    return out


In [13]:
if CLEAN_RENT.exists():
    rent = pd.read_csv(CLEAN_RENT, encoding="utf-8-sig")
    print("기존 정제 CSV 사용:", CLEAN_RENT)
elif RAW_RENT.exists():
    rent_raw = pd.read_csv(RAW_RENT, encoding="utf-8-sig")
    rent = clean_rent(rent_raw)
    rent.to_csv(CLEAN_RENT, index=False, encoding="utf-8-sig")
    print("기존 원본 CSV 정제 완료")
else:
    all_rows = []
    for gu_name, gu_code in seoul_gu.items():
        for year in YEARS:
            for month in range(1, 13):
                ymd = f"{year}{month:02d}"
                month_rows = fetch_rent_month(gu_name, gu_code, ymd)
                all_rows.extend(month_rows)
                print(gu_name, ymd, len(month_rows))
                time.sleep(0.2)
    rent_raw = pd.DataFrame(all_rows)
    rent_raw.to_csv(RAW_RENT, index=False, encoding="utf-8-sig")
    rent = clean_rent(rent_raw)
    rent.to_csv(CLEAN_RENT, index=False, encoding="utf-8-sig")
    print("API 수집 및 정제 완료")

print("전월세 정제 데이터:", rent.shape)
print(rent["dealYear"].value_counts().sort_index())
print(rent["rentType"].value_counts())


기존 정제 CSV 사용: ..\data\processed\seoul_rent_2023_2025_clean.csv
전월세 정제 데이터: (812501, 28)
dealYear
2023    284380
2024    252914
2025    275207
Name: count, dtype: int64
rentType
전세    464333
월세    348168
Name: count, dtype: int64


In [14]:
rent.to_sql("rental_transactions", engine, if_exists="replace", index=False, chunksize=5000)
print("MySQL rental_transactions:", len(rent))

monthly_check = pd.read_sql("""
SELECT guName,
       CONCAT(dealYear, '-', LPAD(dealMonth, 2, '0')) AS yearMonth,
       COUNT(*) AS contractCount
FROM rental_transactions
GROUP BY guName, dealYear, dealMonth
ORDER BY guName, dealYear, dealMonth
""", engine)
print("월별 집계:", monthly_check.shape, monthly_check["yearMonth"].min(), monthly_check["yearMonth"].max())


MySQL rental_transactions: 812501
월별 집계: (900, 3) 2023-01 2025-12


## 2. KOSIS 인구이동

KOSIS 국내인구이동통계(`DT_1B26001_A01`)의 서울 자치구별 월간 총전입·총전출·순이동을 사용합니다. 다운로드 시 **2023.01~2025.12 전체 기간이 포함**되어야 합니다.


In [15]:
kosis_path = RAW / "kosis_migration_2023_2025.csv"
kosis = pd.read_csv(kosis_path, encoding="utf-8", header=[0, 1])

rows = []
for _, row in kosis.iterrows():
    gu = row.iloc[0]
    for year in YEARS:
        for month in range(1, 13):
            ym = f"{year}.{month:02d}"
            rows.append({
                "guName": gu,
                "yearMonth": ym.replace(".", "-"),
                "moveIn": row[(ym, "총전입 (명)")],
                "moveOut": row[(ym, "총전출 (명)")],
                "netMove": row[(ym, "순이동 (명)")]
            })

migration = pd.DataFrame(rows)
for col in ["moveIn", "moveOut", "netMove"]:
    migration[col] = pd.to_numeric(migration[col], errors="coerce")

if len(migration) != 25 * 36:
    raise ValueError(f"KOSIS 결과가 900행이 아닙니다: {len(migration)}")
migration.to_sql("migration", engine, if_exists="replace", index=False)
print("MySQL migration:", migration.shape)


MySQL migration: (900, 5)


## 3. 한국은행 ECOS 기준금리

ECOS `1.3.1. 한국은행 기준금리 및 여수신금리`에서 2023.01~2025.12를 내려받은 CSV를 사용합니다. 월 컬럼명이 `YYYY/MM` 형식인지 확인합니다.


In [16]:
ecos_path = RAW / "ecos_base_rate_2023_2025.csv"
ecos = pd.read_csv(ecos_path, encoding="utf-8")

# 다운로드 파일에서 기준금리 행을 찾을 수 있으면 사용하고, 현재 프로젝트 형식(1행)이면 첫 행을 사용
if "계정항목" in ecos.columns and (ecos["계정항목"] == "한국은행 기준금리").any():
    rate_row = ecos.loc[ecos["계정항목"] == "한국은행 기준금리"].iloc[0]
else:
    rate_row = ecos.iloc[0]

rates = []
for year in YEARS:
    for month in range(1, 13):
        col = f"{year}/{month:02d}"
        if col not in ecos.columns:
            raise KeyError(f"ECOS CSV에 월 컬럼이 없습니다: {col}")
        rates.append({"yearMonth": f"{year}-{month:02d}", "baseRate": rate_row[col]})

interest_rate = pd.DataFrame(rates)
interest_rate["baseRate"] = pd.to_numeric(interest_rate["baseRate"], errors="coerce")
interest_rate.to_sql("interest_rate", engine, if_exists="replace", index=False)
print("MySQL interest_rate:", interest_rate.shape)


MySQL interest_rate: (36, 2)


## 4. 서울 생활인구

서울 열린데이터광장 「자치구 단위 서울 생활인구(내국인)」를 월 평균으로 집계합니다.

**파일 포맷 차이:** 2023 파일은 CP949이며 `기준일ID`, `자치구코드`, `총생활인구수`처럼 한글 컬럼명을 사용합니다. 2024는 UTF-8 영문 컬럼, 2025는 CP949 영문 컬럼이므로 먼저 핵심 3개 컬럼을 동일한 이름으로 맞춘 뒤 결합합니다.


In [17]:
pop_2023 = pd.read_csv(RAW / "LOCAL_PEOPLE_GU_2023.csv", encoding="cp949")
pop_2024 = pd.read_csv(RAW / "LOCAL_PEOPLE_GU_2024.csv", encoding="utf-8")
pop_2025 = pd.read_csv(RAW / "LOCAL_PEOPLE_GU_2025.csv", encoding="cp949")

pop_2023 = pop_2023.rename(columns={
    "기준일ID": "stdr_de_id",
    "자치구코드": "adstrd_code_se",
    "총생활인구수": "tot_lvpop_co"
})
required_pop = ["stdr_de_id", "adstrd_code_se", "tot_lvpop_co"]
for year, frame in [(2023, pop_2023), (2024, pop_2024), (2025, pop_2025)]:
    missing_cols = [c for c in required_pop if c not in frame.columns]
    if missing_cols:
        raise KeyError(f"{year} 생활인구 CSV 컬럼 확인 필요: {missing_cols}")

pop_all = pd.concat([
    pop_2023[required_pop], pop_2024[required_pop], pop_2025[required_pop]
], ignore_index=True)

pop_all["yearMonth"] = (
    pop_all["stdr_de_id"].astype(str).str[:4] + "-" + pop_all["stdr_de_id"].astype(str).str[4:6]
)
pop_all["adstrd_code_se"] = pd.to_numeric(pop_all["adstrd_code_se"], errors="coerce")
pop_all["tot_lvpop_co"] = pd.to_numeric(pop_all["tot_lvpop_co"], errors="coerce")
code_to_gu = {int(code): name for name, code in seoul_gu.items()}
pop_all["guName"] = pop_all["adstrd_code_se"].map(code_to_gu)

living_pop = (
    pop_all.groupby(["guName", "yearMonth"], as_index=False)["tot_lvpop_co"].mean()
    .rename(columns={"tot_lvpop_co": "avgLivingPop"})
)
if len(living_pop) != 25 * 36:
    raise ValueError(f"생활인구 월별 결과가 900행이 아닙니다: {len(living_pop)}")
living_pop.to_sql("living_population", engine, if_exists="replace", index=False)
print("MySQL living_population:", living_pop.shape)


MySQL living_population: (900, 3)


## 5. 4개 데이터 결합 → 머신러닝 테이블

거래 데이터를 `자치구 × 월`로 집계한 뒤 인구이동, 기준금리, 생활인구를 INNER JOIN합니다. 정상적으로 모두 결합되면 **25개 자치구 × 36개월 = 900행**입니다.


In [19]:
ml_data = pd.read_sql("""
SELECT r.guName, r.yearMonth, r.contractCount,
       m.moveIn, m.moveOut, m.netMove,
       i.baseRate, l.avgLivingPop
FROM (
    SELECT guName,
           CONCAT(dealYear, '-', LPAD(dealMonth, 2, '0')) AS yearMonth,
           COUNT(*) AS contractCount
    FROM rental_transactions
    GROUP BY guName, dealYear, dealMonth
) r
JOIN migration m
  ON r.guName = m.guName AND r.yearMonth = m.yearMonth
JOIN interest_rate i
  ON r.yearMonth = i.yearMonth
JOIN living_population l
  ON r.guName = l.guName AND r.yearMonth = l.yearMonth
ORDER BY r.guName, r.yearMonth
""", engine)

print("최종 shape:", ml_data.shape)
print("기간:", ml_data["yearMonth"].min(), "~", ml_data["yearMonth"].max())
print("결측치:")
print(ml_data.isnull().sum())
print("중복 key:", ml_data.duplicated(["guName", "yearMonth"]).sum())

if ml_data.shape[0] != 900:
    raise ValueError(f"최종 데이터가 900행이 아닙니다: {ml_data.shape[0]}")
if ml_data.isna().any().any():
    raise ValueError("최종 데이터에 결측치가 있습니다.")

ml_data.to_sql("ml_rent_market", engine, if_exists="replace", index=False)
print("MySQL ml_rent_market 저장 완료:", len(ml_data))
ml_data.head()


최종 shape: (900, 8)
기간: 2023-01 ~ 2025-12
결측치:
guName           0
yearMonth        0
contractCount    0
moveIn           0
moveOut          0
netMove          0
baseRate         0
avgLivingPop     0
dtype: int64
중복 key: 0
MySQL ml_rent_market 저장 완료: 900


,guName,yearMonth,contractCount,moveIn,moveOut,netMove,baseRate,avgLivingPop
0,강남구,2023-01,2171,6159,6350,-191,3.5,796674.223998
1,강남구,2023-02,2825,9166,8126,1040,3.5,831518.858496
2,강남구,2023-03,3016,11083,8257,2826,3.5,823731.966718
3,강남구,2023-04,2354,7931,6102,1829,3.5,809914.335772
4,강남구,2023-05,2291,7696,6657,1039,3.5,807608.549482


## 6. 산출물

- `data/processed/seoul_rent_2023_2025_clean.csv`
- MySQL `rental_transactions`
- MySQL `migration`
- MySQL `interest_rate`
- MySQL `living_population`
- MySQL `ml_rent_market`

최종 머신러닝 테이블은 서울 25개 자치구 × 36개월(2023-01~2025-12)로 구성되며 총 900행입니다.

> 본 데이터의 `contractCount`는 실제 빈집 수가 아니라 해당 월에 관측된 아파트 전월세 계약 건수입니다.